## Vaccination Campaign Logistics

To best manage the vaccination campaign against seasonal flues, it is intended to make use of Operations Research tools.

The vaccines will arrive from abroad and will initially be stored in a set $D$ of warehouses, from which they must then be distributed to vaccination centers spread over the national territory during a set $S = {1,...,n}$ of weeks. 

Let $J$ be the set of vaccination centers. 
Let $c_{ij}$ be the unit transportation cost of vaccines from warehouse $i$ to vaccination center $j$.
Let $d_i$ be the availability of vaccine packages at warehouse $i$ (having 1 vaccine in warehouse i has a cost $c_i$).
Let $r^s_j$ be the demand for vaccine packages of vaccination center $j$ for week $s$ (based on the bookings made), which must be fully satisfied. 

For logistical reasons, during each week $s$, at most $m^s_i$ vaccine packages can be shipped in total from warehouse $i$ to the vaccination centers. 
Once shipped to a vaccination center, the packages cannot be stored to satisfy the demands of subsequent weeks, due to space constraints. 
Finally, if vaccines are shipped from warehouse $i$ to vaccination center $j$, this must occur for at least 3 consecutive weeks (carrier loyalty constraint).

Write an MILP to help the company to decide the initial allocations of vaccines in each center $d_i$ and how to ship them in the planned horizon to minimize the total costs.

**INDEXES**:
- $i \in D$ warehouse
- $j \in J$ vaccination center
- $s \in S = {1,...,n}$ weeks

**PARAMETERS**:
- $c_{i,j}$ : unit transportation cost from i to j
- $c_i$ : cost of having 1 vaccine packages at warehouse i
- $r^s_i$ : demand for vaccine packages at vaccination center j for week s (must be fully satisfied)
- $m^s_i$ : total vaccine packages that can be shipped from warehouse i in week s

**VARIABLES**:
- $d_i$ : initial allocations of vaccines in warehouse i
- $x_{i,j}^s$ : amount of unit shipped from $i$ to $j$ in week $s$
- $y_{i,j}^s$ : {0,1} 1 if in week $s$ I move packages from $i$ to $j$
- $v_{i,j}^s$ : {0,1} 1 if in week $s$ starting ship from i to j

$$ min \ \ \sum_i d_i c_i + \sum_i \sum_j \sum_s x_{i,j}^s c_{i,j} $$
s.t. $$ \sum_i x_{i,j}^s = r_i^s \ \ \forall j,s $$
$$ \sum_j x_{i,j} \le m_i^s \ \ \forall i,s $$
$$ x_{i,j}^s \le y_{i,j}^s m_i^s $$
$$ \sum_j \sum_s x_{i,j}^s \le d_i \ \ \forall i $$
$$ v_{i,j}^s \ge y_{i,j}^{s} - y_{i,j}^{s-1} $$
$$ y_{i,j}^{s+1} \ge v_{i,j}^s $$
$$ y_{i,j}^{s+2} \ge v_{i,j}^s $$

In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

np.random.seed(42)
m = gp.Model("vaccination_campaign")

num_warehouses = 3
num_centers = 5
num_weeks = 4
D = list(range(num_warehouses))
J = list(range(num_centers))
S = list(range(num_weeks))

c_ij = np.random.uniform(5, 20, (num_warehouses, num_centers))
c_i = np.random.uniform(1, 3, num_warehouses)
r_js = np.random.randint(100, 500, (num_centers, num_weeks))
m_is = np.random.randint(1000, 2000, (num_warehouses, num_weeks))

d = m.addVars(D, vtype=GRB.CONTINUOUS, name="d")
x = m.addVars(D, J, S, vtype=GRB.CONTINUOUS, name="x")
y = m.addVars(D, J, S, vtype=GRB.BINARY, name="y")
v = m.addVars(D, J, S, vtype=GRB.BINARY, name="v")

for j in J:
    for s in S:
        m.addConstr(gp.quicksum(x[i, j, s] for i in D) == r_js[j, s])

for i in D:
    for s in S:
        m.addConstr(gp.quicksum(x[i, j, s] for j in J) <= m_is[i, s])
        
for i in D:
    for j in J:
        for s in S:
            m.addConstr(x[i, j, s] <= y[i, j, s] * m_is[i, s])

for i in D:
    m.addConstr(gp.quicksum(x[i, j, s] for j in J for s in S) <= d[i])

for i in D:
    for j in J:
        for s in S:
            if s > 0:
                m.addConstr(v[i, j, s] >= y[i, j, s] - y[i, j, s-1])
            else:
                m.addConstr(v[i, j, s] >= y[i, j, s])
            
            if s + 1 < num_weeks:
                m.addConstr(y[i, j, s+1] >= v[i, j, s])
            if s + 2 < num_weeks:
                m.addConstr(y[i, j, s+2] >= v[i, j, s])

m.setObjective(
    gp.quicksum(d[i] * c_i[i] for i in D) + 
    gp.quicksum(x[i, j, s] * c_ij[i, j] for i in D for j in J for s in S), 
    GRB.MINIMIZE
)
m.optimize()

if m.status == GRB.OPTIMAL:
    print(f"Total Minimum Cost: €{m.objVal:.2f}")
    for i in D:
        print(f"Warehouse {i} initial allocation: {d[i].x:.0f} vaccines")


Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))
CPU model: 13th Gen Intel(R) Core(TM) i7-13620H, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 16 logical processors, using up to 16 threads
Optimize a model with 230 rows, 183 columns and 618 nonzeros (Min)
Model fingerprint: 0x96ab1697
Model has 63 linear objective coefficients
Variable types: 63 continuous, 120 integer (120 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [1e+00, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+02, 2e+03]
Found heuristic solution: objective 85190.445813
Presolve removed 230 rows and 183 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 16 available processors)
Solution count 2: 58562.5 85190.4 
Optimal solution found (tolerance 1.00e-04)
Best objective 5.856254918854e+04, best bound 5.8